In [ ]:
PROJECT_RANDOM_STATE = 2627
print("PROJECT_RANDOM_STATE =", PROJECT_RANDOM_STATE)

# Practice 3 — Challenge A: multinomial logistic regression (R2)

**Scope:** P3.1, P3.2 and the Challenge A part of P3.5.

> **Team code style (delete this cell before submitting).** The team must be able to defend every line.
>
> - Prefer tools seen in class or named in the statement. If you cannot explain a line in one sentence, rewrite it.
> - Plain `for` loops and `if`s, one step per line, clear variable names. No lambdas, classes or long chains.
> - Same results, simpler code. After simplifying, restart, run all and check that the numbers did not change.
> - Interpretation goes in Markdown cells, in English. Check your notebook with `revisor_notebook.py`.
>
> **Shared names, so that the notebooks can be joined:** `dev` (development table), `X`, `y` (cyber target), `z` (storage target), `latency_ns`, `NOMINAL`, `NUMERIC`, `FEATURES`, `REFERENCE_LEVELS`, `make_preprocessor()`, `folds` (list of `(train_idx, val_idx)`), `CLASS_NAMES` (cyber).

## Setup from R1 and R4

Paste here the cells published by R1 and R4. They are **not** pasted into the final notebook.

In [ ]:
# TODO: paste the data, preprocessing and folds cells
# TEMPORARY setup

# Standard library
import json
import time

# Numerical and table libraries
import numpy as np
import pandas as pd 

# Plotting library
import matplotlib.pyplot as plt

# scikit-learn tools to build the preprocessing block
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler

# Folder where the cyber data live
DATA_DIR = "../datos/cyber/"

# Development table: one row per network connection
dev = pd.read_csv(DATA_DIR + "cyber_development.csv")

# Read the file that maps each class code to its name
with open(DATA_DIR + "cyber_class_codes.json") as fh:
    raw_codes = json.load(fh)

# JSON keys are always text, so we convert each key to an integer
CLASS_NAMES = {}
for key in raw_codes:
    CLASS_NAMES[int(key)] = raw_codes[key]

# Categorical predictors
NOMINAL = ["proto", "service", "conn_state"]

# Numeric predictors
NUMERIC = ["duration", "src_bytes", "dst_bytes", "missed_bytes", "src_pkts", "src_ip_bytes", "dst_pkts", "dst_ip_bytes"]

# All predictors together: categorical first, then numeric
FEATURES = NOMINAL + NUMERIC

# Predictor table and cyber target
X = dev[FEATURES]
y = dev["attack_code"]

# PLACEHOLDER reference levels: R1 publishes the real ones
REFERENCE_LEVELS = {"proto": "tcp", "service":"-", "conn_state": "SF"}

def make_preprocessor():
    # Numeric columns: first log1p, then standardise
    numeric_steps = Pipeline([
        ("log1p", FunctionTransformer(np.log1p, feature_names_out="one-to-one")),
        ("scale", StandardScaler()),
    ])

    # List of the level to drop for each categorical column, in the same order as NOMINAL
    drop_list = []
    for col in NOMINAL:
        drop_list.append(REFERENCE_LEVELS[col])

    # One-hot encoding without the reference level; unseen levels become all zeros
    nominal_steps = OneHotEncoder(drop=drop_list, handle_unknown="ignore")

    # Apply each block to its columns; sparse_threshold=0 returns a normal (dense) array
    return ColumnTransformer([
        ("nominal", nominal_steps, NOMINAL),
        ("numeric", numeric_steps, NUMERIC),
    ], sparse_threshold=0)

# Canonical folds file: says which fold each row belongs to
canonical = pd.read_csv(DATA_DIR + "cyber_folds.csv")

# Safety check: the folds file must be in the same row order as the development table
assert (canonical["row_id"] == dev["row_id"]).all()

# Fold number of each row, as a NumPy array
fold_id = canonical["fold_id"].to_numpy()

# Build the list of (train_idx, val_idx) pairs, one pair per fold
folds = []
for f in range(5):
    # Training rows: all rows that are NOT in the fold f
    train_idx = np.where(fold_id != f)[0]
    # Validation rows: the rows that ARE in fold f
    val_idx = np.where(fold_id == f)[0]
    # Store the pair
    folds.append((train_idx, val_idx))

# P3.1 — Multinomial logistic regression

## Two different things are called `C`
The course uses $C$ for the **number of classes**. The `scikit-learn` parameter `C` means something unrelated: the **inverse regularisation strength**, where a larger value means a weaker penalty. Following the Unit 7 notebook, the library parameter is always passed from a named constant, `C_SKLEARN_UNREGULARISED`, so the two are never confused. With `C = inf` the penalty term disappears completely, which is what p3.1 asks for.

In [ ]:
# TODO: imports for this part
# Multinomial logistic regression
from sklearn.linear_model import LogisticRegresson

# K nearest neighbours
from sklearn.neighbors import KNeighborsClassifier

# Metrics: accuracy, F1, confusion matrix and per-class precision/recall/F1/support
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score
from sklearn.metrics import precision_recall_fscore_support

# Fixed settings, written once here before any model is built

# scikit-learn's C is the INVERSE regularisation strength, NOT the number of classes
# Infinity means the penalty term is removed completely
C_SKLEARN_UNREGULARISED = np.inf
# Maximum number of solver iterations: only so the solver can finish, not a modelling choice
MAX_ITER = 5000
# Tolerance for the numerical checks (obtained from Unit 7 notebook)
ATOL_PRED = 1e-9

In [ ]:
# TODO: model on the required route of Section 2, inside a Pipeline.
# Print the effective penalty parameters from get_params().

def make_logreg():
    # Each call returns a NEW, untrained pipeline: R1's preprocessing + logistic regression
    return Pipeline([
        # Step 1: the preprocessing block from R1
        ("prep", make_preprocessor()),
        # Step 2: C = infinity means no penalty, no class weights, enough iterations to converge
        ("logreg", LogisticRegression(
            C=C_SKLEARN_UNREGULARISED,
            solver = "lbfgs",
            class_weight=None, 
            max_iter=MAX_ITER,
        )),
    ])

# Build one pipeline only to read its parameters
logreg = make_logreg()

# get_params() returns a dictionary with every parameter of the logistic regression step
params = logreg.named_steps["logreg"].get_params()

# Print the parameters that show there is no penalty
for name in ["C", "l1_ratio", "penalty", "class_weight", "solver", "max_iter"]:
    # Some names may not exist in every scikit-learn version, so we check first
    if name in params:
        print(name, "=", params[name])

In [ ]:
# TODO: macro F1 for each of the five folds, its mean and spread, weighted F1 and accuracy
# Class codes in increasing order:
CLASS_CODES = sorted(CLASS_NAMES)

# Class names in the same order as CLASS_CODES
class_names = []
for code in CLASS_CODES:
    class_names.append(CLASS_NAMES[code])

# Target as a NumPy array, to index it by position
y_true = y.to_numpy()

# Out-of-fold predictions: each row will be predicted by the model that did NOT train on it
oof_pred = np.zeros(len(y_true), dtype=int)

# One dictionary of results per fold
lr_rows = []

# One dictionary per fold with the shape of coef_ and the gssapi count
lr_coef_info = []

# Loop over the five canonical folds; k is the fold number
for k, (train_idx, val_idx) in enumerate(folds):
    # New untrained pipeline for this fold
    model = make_logreg()

    # Start the clock
    t0 = time.perf_counter()
    # Train on the training rows of this fold
    model.fit(X.iloc[train_idx], y_true[train_idx])
    # Seconds spent fitting
    fit_time = time.perf_counter() - t0

    # Check that the model saw all ten classes, in the order of CLASS_CODES
    assert list(model.classes_) == CLASS_CODES

    # Predict the validation rows
    pred = model.predict(X.iloc[val_idx])
    # Save these predictions in their positions of the out-of-fold array
    oof_pred[val_idx] = pred

    # The logistic regression step, to read coef_ and n_iter_
    lr = model.named_steps["logreg"]

    # Metrics of this fold
    lr_rows.append({
        "fold": k,
        "macro_f1": f1_score(y_true[val_idx], pred, average="macro"),
        "weighted_f1": f1_score(y_true[val_idx], pred, average="weighted"),
        "accuracy": accuracy_score(y_true[val_idx], pred),
        "fit_time_s": fit_time,
        # n_iter_ is an array with one value for the multinomial model
        "iterations": lr.n_iter_[0],
    })

    # How many training rows have service = gssapi in this fold
    gssapi_count = (X.iloc[train_idx]["service"] == "gssapi").sum()

    # Store the shape of coef_ beside that count
    lr_coef_info.append({
        "fold": k,
        "coef_shape": lr.coef_.shape,
        "gssapi_rows_in_training": gssapi_count,
    })

# Turn the list of dictionaries into a table, one row per fold
lr_folds = pd.DataFrame(lr_rows).set_index("fold")
display(lr_folds.round(4))

# Macro F1 column, used several times below
lr_macro = lr_folds["macro_f1"]

# Mean of the five folds 
print("Macro F1 mean:", round(lr_macro.mean(), 4))
# Spread as range: best fold minus worst fold
print("Macro F1 spread (max - min):", round(lr_macro.max() - lr_macro.min(), 4))
# Spread as standard deviation of the five values
print("Macro F1 std:", round(lr_macro.std(ddof=0), 4))
# Means of the other two metrics
print("Weighted F1 mean:", round(lr_folds["weighted_f1"].mean(), 4))
print("Accuracy mean:", round(lr_folds["accuracy"].mean(), 4))
# True if no fit used the whole iteration budget (that is, every fit converged)
print("All fits converged before max_iter:", (lr_folds["iterations"] < MAX_ITER).all())

**About the warning above.** On recent versions of `scikit-learn`, fitting with `C = inf` may print *"Setting penalty=None will ignore the C and l1_ratio parameters"*. This warning is expected: with the penalty term removed, `C`and `l1_ratio` no longer play any role, which is exactly what P3.1 asks for.

In [1]:
# TODO: per-class precision, recall and F1 beside their support, and the labelled confusion matrix
# Per-class metrics computed on the out-of-fold predictions (each row predicted by a model
# that did not train on it). labels=CLASS_CODES fixes the order of the classes.
# zero_division=0: if a class is never predicted, its precision is 0 instead of an error
precision, recall, f1, support = precision_recall_fscore_support(y_true, oof_pred, labels=CLASS_CODES, zero_division=0)

# One row per class, with its support beside the three metrics
per_class = pd.DataFrame({
    "precision": precision,
    "recall": recall,
    "f1": f1,
    "support": support,
}, index = class_names)

# Sort by support so the rare classes are easy to find at the bottom
per_class = per_class.sort_values("support", ascending=False)
display(per_class.round(4))

# Confusion matrix: rows = true class, columns = predicted class, same order as CLASS_CODES
cm = confusion_matrix(y_true, oof_pred, labels=CLASS_CODES)

# Labels for rows and columns, so the table can be read without remembering the codes
row_labels = []
col_labels = []
for name in class_names:
    row_labels.append("true: " + name)
    col_labels.append("pred: " + name)

cm_table = pd.DataFrame(cm, index=row_labels, columns=col_labels)
display(cm_table)

# Safety check: the matrix must count every development row exactly once
assert cm.sum() == len(y_true)

**Interpretation.** TODO

In [ ]:
# TODO: print the shape of coef_
# Shape of coef_ in each fold, beside how many training rows had service = gssapi
lr_coef_table = pd.DataFrame(lr_coef_info).set_index("fold")
display(lr_coef_table)

# Final model: the same pipeline fitted once on ALL development rows
# It is only used to read and interpret the coefficients in P3.2
final_model = make_logreg()
final_model.fit(X, y_true)

# The logistic regression step of the final model
final_lr = final_model.named_steps["logreg"]

# Check the class order, as in every fold
assert list(final_lr.classes_) == CLASS_CODES

print("coef_ shape of the final model:", final_lr.coef_.shape)
print("intercept_ chape of the final model:", final_lr.intercept_.shape)
print("number of transformed predictors:", len(final_model.named_steps["prep"].get_feature_names_out()))


**What the shape of `coef_` implies.** 

`coef_` has shape ([10],[p]): one row per class and one column per transformed predictor (the dummy columns without their reference levels, plus the [8] standardised numeric columns). There are $C$ rows, not the $C - 1$ of the reference-class form, so a single row cannot be read on its own: adding the same vector to every row would give exactly the same probabilities. [If the shape changes between folds:] In fold(s) [k] the training rows contained no `service = gssapi`, so the encoder created one column fewer and the coefficients of different folds cannot be compared column by column.

# P3.2 — The parameterisation bridge

**Reference class and why.** 

We use `normal` as the reference class. Every differenced coefficient then reads as "this attack versus normal traffic", which is the question that matters for detecting attacks. The choice does not change the model: subtracting the same row from every class leaves all probabilities identical, so it only changes how the coefficients are written down.

In [ ]:
# TODO: construct w_c and b_c (Section 3)
# Look up the codes of "normal" and "ransomware" by their names
REFERENCE_CLASS = None
RANSOMWARE = None
for code in CLASS_NAMES:
    if CLASS_NAMES[code] == "normal":
        REFERENCE_CLASS = code
    if CLASS_NAMES[code] == "ransomware":
        RANSOMWARE = code

# Name of the reference class, to use in titles and sentences
REFERENCE_NAME = CLASS_NAMES[REFERENCE_CLASS]

# Number of ransomware rows
RANSOMWARE_SUPPORT = (y_true == RANSOMWARE).sum()
print("reference class:", REFERENCE_NAME, "| ransomware support:", RANSOMWARE_SUPPORT)

# Stored parameterisation: one row of coefficients per class
beta = final_lr.coef_
# Stored intercepts: one per class
alpha = final_lr.intercept_

# Position of the reference class inside the rows of coef_
r_pos = list(final_lr.classes_).index(REFERENCE_CLASS)

# Reference-class parameterisation: subtract the reference row from every row
w = beta - beta[r_pos]
# Same for the intercepts
b = alpha - alpha[r_pos]

# Stop the notebook if the reference row is not all zeros
assert np.all(w[r_pos] == 0)
# Stop the notebook if the reference intercept is not zero
assert b[r_pos] == 0
print("reference row of w and reference intercept b_r are exactly zero")

# Column names produced by the preprocessor
names = final_model.named_steps["prep"].get_feature_names_out()

# Table of differenced coefficients: rows = predictors, columns = classes
coef_diff = pd.DataFrame(w.T, index = names, columns = class_names)
display(coef_diff.round(3))

In [ ]:
# TODO: verify the Topic 7 probabilities against predict_proba for at least one observation
# Transform all rows with the fitted preprocessor
X_trans = final_model.named_steps["prep"].transform(X)

# If the preprocessor returns a sparse matrix, turn it into a normal NumPy array
if not isinstance(X_trans, np.ndarray):
    X_trans = X_trans.toarray()

# One observation step by step: the first ransomware row

# Position of the first row whose true class is ransomware
row = np.where(y_true == RANSOMWARE)[0][0]

# The transformed predictors of that row
x_row = X_trans[row]

# Topic 7 score of each class: b_c + w_c · x (the @ sign is the dot product)
scores = b + w @ x_row

# Subtract the largest score: probabilities do not change, and np.exp cannot overflow
scores = scores - scores.max()

# Softmax: exponentiate each score and divide by their sum
proba_topic7 = np.exp(scores) / np.exp(scores).sum()

# Probabilities given by scikit-learn for the same row
proba_sklearn = final_model.predict_proba(X.iloc[[row]])[0]

# Show both columns side by side
check = pd.DataFrame(
    {"topic 7 (w, b)": proba_topic7, "predict_proba": proba_sklearn},
    index=class_names)
print("row_id:", dev["row_id"].iloc[row], "| true class:", CLASS_NAMES[y_true[row]])
display(check.round(6))

# Stop the notebook if both routes disagree by more than the tolerance
assert np.allclose(proba_topic7, proba_sklearn, atol=ATOL_PRED, rtol=0.0)

# The same check for one row of every class
for code in CLASS_CODES:
    # First row of this class
    row = np.where(y_true == code)[0][0]
    # Topic 7 probabilities, exactly as above
    scores = b + w @ X_trans[row]
    scores = scores - scores.max)=
    proba_topic7 = np.exp(scores) / np.exp(scores).sum()
    # scikit-learn probabilities
    proba_sklearn = final_model.predict_proba(X.iloc[[row]])[0]
    # Largest absolute difference between the two
    max_diff = np.abs(proba_topic7 - proba_sklearn).max()
    print(CLASS_NAMES[code], "| max difference:", max_diff)
    # Stop the notebook if the two probabilities disagree for this class
    assert max_diff <= ATOL_PRED

print("topic 7 probabilities match predict_proba for one row of every class")

**The three parameterisations.**

1. The model's probabilities: TODO
2. scikit-learn's stored parameterisation: TODO
3. The reference-class parameterisation: TODO

In [ ]:
# StandardScaler fitted inside the final model 
scaler = final_model.named_steps["prep"].named_transformers_["numeric"].named_steps["scale"]

# Standard deviation of log1p(x) for each numeric column, in the order of NUMERIC
LOG1P_SD = {}
for i in range(len(NUMERIC)):
    LOG1P_SD[NUMERIC[i]] = scaler.scale_[i]

# ------- Numeric predictor
class_name = "ddos"
col = "src_bytes"
# Odds ratio: exp of the differenced coefficient 
ratio = np.exp(coef_diff.loc[col, class_name])
# One standard deviation of log1p(x) mean multiplying (1 + x) by this factor
factor = np.exp(LOG1P_SD[col])
print("Holding the other predictors fixed, one standard deviation more in log1p(" + col + ")",
      "(multiplying 1 +", col, "by", round(factor, 2), ")",
      "multiplies the odds of", class_name, "versus", REFERENCE_NAME, "by", round(ratio, 3))

# -------- Categorical predictor
class_name = "scanning"
col = "conn_state"
level = "REJ"
# The dummy column for this level is called "conn_state_REJ"
ratio = np.exp(coef_diff.loc[col + "_" + level, class_name])
print("Holding the other predictors fixed, a connection with", col, "=", level,
      "instead of the reference level", col, "=", REFERENCE_LEVELS[col],
      "has its odds of", class_name, "versus", REFERENCE_NAME,
      "multiplied by", round(ratio, 3))

# --------- Rare class, beside its support
class_name = "ransomware"
col = "dst_bytes"
ratio = np.exp(coef_diff.loc[col, class_name])
print("Holding the other predictors fixed, one standard deviation more in log1p(" + col + ")",
      "multiplies the odds of ransomware (support:", RANSOMWARE_SUPPORT, ")",
      "versus", REFERENCE_NAME, "by", round(ratio, 3))

print()
print("These are associations in the fitted model, not causal effects.")

**Odds ratios (Section 3 rule).**

- Numeric predictor: TODO
- Categorical predictor: TODO
- Rare class, beside its support: TODO

These are associations in the fitted model, not causal effects. The model has no penalty, so very large or very small odds ratios (especially for rare classes) can appear simply because nothing limits the size of the coefficients. They do not by themselves prove perfect separation.

In [ ]:
# TODO: the required coefficient plot
# Remove the reference column: it is all zeros and adds nothing to the figure
plot_table = coef_diff.drop(columns=REFERENCE_NAME)

# Symmetric colour limits, so that 0 is always the white centre of the colour map
limit = np.abs(plot_table.to_numpy()).max()

fig, ax = plt.subplots(figsize=(10,9))
# Red = positive coefficient, blue = negative, white = close to zero
image = ax.imshow(plot_table.to_numpy(), cmap="RdBu_r", vmin=-limit, vmax=limit, aspect="auto")

# Class names on the x axis, predictor names on the y axis
ax.set_xticks(range(len(plot_table.columns)))
ax.set_xticklabels(plot_table.columns, rotation=45, ha="right")
ax.set_yticks(range(len(plot_table.index)))
ax.set_yticklabels(plot_table.index)

fig.colorbar(image, ax=ax, label="differenced coefficient (log-odds versus " + REFERENCE_NAME + ")")
ax.set_title("Reference-class coefficients of the final model (reference: " + REFERENCE_NAME + ")")
fig.tight_layout()
plt.show()

**Interpretation of the figure.** TODO

# P3.5 — Challenge A against Practice 2

In [ ]:
# TODO: the paired per-fold differences
# Macro F1 of the Practice 2 model on the same five canonical folds
P2_MACRO_F1 = []

# Stop the notebook until the five values are pasted
assert len(P2_MACRO_F1) == 5
assert not np.isnan(P2_MACRO_F1).any(), "paste the five Practice 2 values first"

# One row per fold: both models side by side
paired = pd.DataFrame({
    "practice_2": P2_MACRO_F1,
    "logreg": lr_macro.to_numpy(),
}, index=lr_macro.index)

# Paired difference: positive means logistic regression did better in that fold
paired["difference"] = paired["logreg"] - paired["practice_2"]
display(paired.round(4))

# Descriptive summary of the five differences
print("mean difference:", round(paired["difference"].mean(), 4))
print("smallest difference:", round(paired["difference"].min(), 4))
print("largest difference:", round(paired["difference"].max(), 4))
print("folds where logistic regression is better:", (paired["difference"] > 0).sum(), "of 5")

### The six fixed comparison questions

1. **Size and consistency of the change.** Using the five paired differences (descriptive only). TODO
2. **Route and representation.** Pipeline comparison or isolated family comparison, and what the evidence licenses. TODO
3. **What can the new family express** that the previous one could not? TODO
4. **What did it cost** in fitting time, tuning effort and interpretability? TODO
5. **Did the ordering of the two challenges' difficulty change**, and why? TODO
6. **Would you ship the new model?** If not, what would change your mind? TODO